# 🎭 Customer Sentiment Analysis

**Goal:** Classify tweets into `positive`, `negative`, or `neutral` sentiment using machine learning.

**Pipeline:**
1. Exploratory Data Analysis (EDA)
2. Text Preprocessing
3. Feature Engineering (TF-IDF)
4. Model Training (Logistic Regression & Random Forest)
5. Experiment Tracking with MLflow
6. Deployment via Streamlit

**Dataset:** ~27,000 tweets with demographic metadata (country, age group, time of tweet)

## 1. Setup & Imports

In [ ]:
# Install dependencies
!pip install mlflow scikit-learn pandas matplotlib seaborn wordcloud joblib --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import os
import joblib

from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

import mlflow
import mlflow.sklearn

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

print("All libraries loaded successfully ✅")

## 2. Load Raw Data

In [ ]:
train_raw = pd.read_csv('train.csv', encoding='ISO-8859-1')
test_raw  = pd.read_csv('test.csv',  encoding='ISO-8859-1')

print(f"Train shape: {train_raw.shape}")
print(f"Test shape : {test_raw.shape}")
train_raw.head()

In [ ]:
# Basic info
print("=== Train Data Info ===")
train_raw.info()
print("\nMissing values:")
print(train_raw.isnull().sum())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Sentiment Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sentiment_counts = train_raw['sentiment'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#3498db']

axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Sentiment Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for i, v in enumerate(sentiment_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Sentiment Share', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable Overview', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Sentiment by Age Group ---
age_sentiment = train_raw.groupby(['Age of User', 'sentiment']).size().unstack(fill_value=0)

age_sentiment.plot(kind='bar', figsize=(10, 5), color=['#e74c3c', '#3498db', '#2ecc71'],
                   edgecolor='white', linewidth=0.8)
plt.title('Sentiment by Age Group', fontsize=14, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()

In [ ]:
# --- Sentiment by Time of Tweet ---
time_sentiment = train_raw.groupby(['Time of Tweet', 'sentiment']).size().unstack(fill_value=0)

time_sentiment.plot(kind='bar', figsize=(9, 5), color=['#e74c3c', '#3498db', '#2ecc71'],
                    edgecolor='white', linewidth=0.8)
plt.title('Sentiment by Time of Tweet', fontsize=14, fontweight='bold')
plt.xlabel('Time of Day')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()

In [ ]:
# --- Text Length Distribution by Sentiment ---
train_raw['text_length'] = train_raw['text'].dropna().apply(len)

plt.figure(figsize=(10, 5))
for label, color in zip(['positive', 'negative', 'neutral'], ['#2ecc71', '#e74c3c', '#3498db']):
    subset = train_raw[train_raw['sentiment'] == label]['text_length'].dropna()
    subset.plot(kind='hist', bins=40, alpha=0.6, color=color, label=label)

plt.title('Tweet Length Distribution by Sentiment', fontsize=14, fontweight='bold')
plt.xlabel('Character Count')
plt.ylabel('Frequency')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()

In [ ]:
# --- Word Clouds ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, label, cmap in zip(axes,
                            ['positive', 'negative', 'neutral'],
                            ['Greens', 'Reds', 'Blues']):
    text = ' '.join(train_raw[train_raw['sentiment'] == label]['text'].dropna())
    wc = WordCloud(width=500, height=300, background_color='white',
                   colormap=cmap, max_words=100).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{label.capitalize()} Tweets', fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle('Word Clouds by Sentiment', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Text Preprocessing

The preprocessing pipeline applied to create `cleaned_text`:
- Lowercase conversion
- Remove URLs, mentions (`@user`), hashtags
- Remove punctuation and digits
- Strip extra whitespace

In [ ]:
def clean_text(text: str) -> str:
    """Clean and normalize a tweet string."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # remove URLs
    text = re.sub(r'@\w+', '', text)                      # remove @mentions
    text = re.sub(r'#\w+', '', text)                      # remove hashtags
    text = re.sub(r'[^\x00-\x7F]+', '', text)            # remove non-ASCII
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)                       # remove digits
    text = re.sub(r'\s+', ' ', text).strip()             # normalize spaces
    return text

# Apply preprocessing
train_raw['cleaned_text'] = train_raw['text'].apply(clean_text)
test_raw['cleaned_text']  = test_raw['text'].apply(clean_text)

# Preview
train_raw[['text', 'cleaned_text', 'sentiment']].head(5)

In [ ]:
# Save cleaned data
train_raw.to_csv('cleaned_train.csv', index=False, encoding='utf-8')
test_raw.to_csv('cleaned_test.csv',   index=False, encoding='utf-8')
print("Cleaned data saved ✅")

## 5. Feature Engineering & Train/Validation Split

In [ ]:
# Load cleaned data (or use the dataframes above)
train_data = pd.read_csv('cleaned_train.csv', encoding='ISO-8859-1')
train_data = train_data.dropna(subset=['cleaned_text', 'sentiment'])

X = train_data['cleaned_text']
y = train_data['sentiment']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {len(X_train):,}")
print(f"Val size   : {len(X_val):,}")

In [ ]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)

# Scale (required for Logistic Regression with sparse matrices)
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train_tfidf)
X_val_scaled   = scaler.transform(X_val_tfidf)

print(f"Feature matrix shape: {X_train_scaled.shape}")

## 6. Model Training & Evaluation

In [ ]:
# --- Logistic Regression ---
log_reg = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_val_scaled)

print("=== Logistic Regression ===")
print(classification_report(y_val, y_pred_lr))

In [ ]:
# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_val_scaled)

print("=== Random Forest ===")
print(classification_report(y_val, y_pred_rf))

In [ ]:
# --- Confusion Matrices (side-by-side) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in zip(axes,
                              [y_pred_lr, y_pred_rf],
                              ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_val, preds, labels=['negative', 'neutral', 'positive'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['negative', 'neutral', 'positive'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Experiment Tracking with MLflow

In [ ]:
# Use env variable for tracking URI (override with MLFLOW_TRACKING_URI if needed)
tracking_uri = os.getenv('MLFLOW_TRACKING_URI', 'file:./mlruns')
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment('customer_sentiment_analysis')

def log_model_run(model, model_name, X_train, X_val, y_train, y_val, vectorizer, scaler):
    with mlflow.start_run(run_name=model_name) as run:
        # Log preprocessors
        joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')
        joblib.dump(scaler,     'standard_scaler.joblib')
        mlflow.log_artifact('tfidf_vectorizer.joblib', 'preprocessors')
        mlflow.log_artifact('standard_scaler.joblib',  'preprocessors')

        # Train & predict
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        report = classification_report(y_val, preds, output_dict=True)

        # Log params & metrics
        mlflow.log_param('model_type', model_name)
        mlflow.log_metric('accuracy',  model.score(X_val, y_val))
        mlflow.log_metric('precision', report['macro avg']['precision'])
        mlflow.log_metric('recall',    report['macro avg']['recall'])
        mlflow.log_metric('f1_score',  report['macro avg']['f1-score'])

        # Log model
        signature = mlflow.models.infer_signature(X_val, preds)
        mlflow.sklearn.log_model(model, model_name.lower().replace(' ', '_'),
                                 signature=signature)

        print(f"[{model_name}] Run ID: {run.info.run_id}")
        print(classification_report(y_val, preds))
        return run.info.run_id

lr_run_id = log_model_run(
    LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Logistic Regression',
    X_train_scaled, X_val_scaled, y_train, y_val,
    vectorizer, scaler
)

rf_run_id = log_model_run(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Random Forest',
    X_train_scaled, X_val_scaled, y_train, y_val,
    vectorizer, scaler
)

# Save run IDs for reference
with open('mlflow_run_ids.txt', 'w') as f:
    f.write(f'logistic_regression: {lr_run_id}\n')
    f.write(f'random_forest: {rf_run_id}\n')

print("\nAll runs logged ✅")

In [ ]:
# Launch MLflow UI (run in terminal instead for non-Colab environments)
# !mlflow ui --port 5000
print("To view the MLflow dashboard, run in your terminal:")
print("  mlflow ui --port 5000")
print("Then open: http://localhost:5000")

## 8. Save Final Model

In [ ]:
# Re-train best model on full training data and save
best_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
best_model.fit(X_train_scaled, y_train)

os.makedirs('models', exist_ok=True)
joblib.dump(best_model,  'models/sentiment_model.joblib')
joblib.dump(vectorizer,  'models/tfidf_vectorizer.joblib')
joblib.dump(scaler,      'models/standard_scaler.joblib')

print("Model artifacts saved to models/ ✅")

## 9. Inference Example

In [ ]:
def predict_sentiment(text: str) -> str:
    """Predict sentiment for a single tweet."""
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    scaled = scaler.transform(vec)
    return best_model.predict(scaled)[0]

examples = [
    "I absolutely love this product! Amazing experience 🎉",
    "Terrible service, would never buy again.",
    "The package arrived today.",
]

for tweet in examples:
    print(f"  Tweet    : {tweet}")
    print(f"  Sentiment: {predict_sentiment(tweet)}")
    print()